# Colab training notebook

Notebook nay dung de train News-Article-Recommendation-System voi MINDsmall tren Google Colab.

Truoc khi chay: vao `Runtime` -> `Change runtime type` -> chon `GPU` neu co.


In [ ]:
# Check GPU runtime. Neu khong co GPU, training van chay CPU nhung se cham hon.
import subprocess

try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception as exc:
    print("No GPU detected. Use Runtime > Change runtime type > GPU if available.")


## 1. Mount Google Drive

Dat MINDsmall trong Google Drive. Folder co the la folder chua truc tiep `news.tsv`/`behaviors.tsv`, hoac folder cha co `MINDsmall_train`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Clone or update repository


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ThienTan142/News-Article-Recommendation-System.git"
BRANCH = "codex/mindsmall-pipeline-cleanup"
PROJECT_DIR = Path("/content/News-Article-Recommendation-System")
VENV_DIR = Path("/content/news-rec-venv")
PYTHON_BIN = VENV_DIR / "bin" / "python"
RESET_VENV = True

def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

if PROJECT_DIR.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run(["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_DIR])

print("Project dir:", PROJECT_DIR)


## 3. Install dependencies

Colab co san nhieu binary package. Neu `numpy` va `pandas` bi lech ABI, import se loi `numpy.dtype size changed`. Cell nay tao virtualenv rieng va chay toan bo pipeline bang Python trong virtualenv de tranh package mac dinh cua Colab. `RESET_VENV = True` se xoa moi virtualenv cu trong `/content/news-rec-venv` de tranh giu lai package loi tu lan chay truoc.


In [ ]:
COLAB_COMPAT_PACKAGES = [
    "numpy==1.26.4",
    "pandas==2.1.4",
    "scipy==1.11.4",
    "scikit-learn==1.4.2",
]

if RESET_VENV and VENV_DIR.exists():
    print("Removing old virtualenv:", VENV_DIR)
    shutil.rmtree(VENV_DIR)

if not PYTHON_BIN.exists():
    run([sys.executable, "-m", "venv", VENV_DIR])

run([PYTHON_BIN, "-m", "pip", "install", "--quiet", "--upgrade", "pip", "setuptools", "wheel"])
run([
    PYTHON_BIN,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--force-reinstall",
    "--no-cache-dir",
    *COLAB_COMPAT_PACKAGES,
], cwd=PROJECT_DIR)
run([PYTHON_BIN, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], cwd=PROJECT_DIR)

version_check = """
import numpy as np
import pandas as pd
import scipy
import sklearn
print('python', __import__('sys').executable)
print('numpy_path', np.__file__)
print('pandas_path', pd.__file__)
print('numpy', np.__version__)
print('pandas', pd.__version__)
print('scipy', scipy.__version__)
print('sklearn', sklearn.__version__)
print('pandas_smoke', pd.DataFrame({'ok': [1]}).to_dict())
"""
run([PYTHON_BIN, "-c", version_check], cwd=PROJECT_DIR)
run([PYTHON_BIN, "-m", "pip", "check"], cwd=PROJECT_DIR)


## 4. Configure MINDsmall path

Sua `MIND_DIR` cho dung vi tri dataset cua ban tren Google Drive.

Vi du hop le:

- `/content/drive/MyDrive/MINDsmall`
- `/content/drive/MyDrive/Project/data/MINDsmall`
- `/content/drive/MyDrive/MINDsmall/MINDsmall_train`


In [ ]:
MIND_DIR = "/content/drive/MyDrive/MINDsmall"

resolve_code = """
import sys
from src.mind_dataset import resolve_mindsmall_paths
paths = resolve_mindsmall_paths(sys.argv[1])
print('Resolved train dir:', paths.root)
print('News:', paths.news_path)
print('Behaviors:', paths.behaviors_path)
"""
run([PYTHON_BIN, "-c", resolve_code, MIND_DIR], cwd=PROJECT_DIR)


## 5. Build precompute artifacts

Cell `precompute_news.py` co the mat vai phut vi phai encode toan bo news bang SentenceTransformer.


In [ ]:
run([PYTHON_BIN, "scripts/precompute_news.py", "--mind-dir", MIND_DIR], cwd=PROJECT_DIR)
run([PYTHON_BIN, "scripts/precompute_user_history.py", "--mind-dir", MIND_DIR], cwd=PROJECT_DIR)


## 6. Build CTR dataset


In [ ]:
run([PYTHON_BIN, "scripts/build_ctr_dataset.py", "--mind-dir", MIND_DIR, "--neg-ratio", "4"], cwd=PROJECT_DIR)


## 7. Train CTR reranker

Mac dinh duoi day dung cau hinh full hon local smoke test. Neu Colab het RAM/VRAM, giam `BATCH_SIZE` xuong `512` hoac `256`.


In [ ]:
# Set TRAIN_MAX_ROWS to an integer for a fast smoke run, or None to train all rows.
TRAIN_MAX_ROWS = None
EPOCHS = 8
BATCH_SIZE = 1024

device_check = subprocess.run(
    [str(PYTHON_BIN), "-c", "import torch; print('cuda' if torch.cuda.is_available() else 'cpu')"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
    check=True,
)
env = os.environ.copy()
env["NEWS_REC_DEVICE"] = device_check.stdout.strip()
print("Training device:", env["NEWS_REC_DEVICE"])

train_args = [
    PYTHON_BIN,
    "-m",
    "src.train",
    "--epochs",
    EPOCHS,
    "--batch-size",
    BATCH_SIZE,
]
if TRAIN_MAX_ROWS is not None:
    train_args.extend(["--max-rows", TRAIN_MAX_ROWS])

run(train_args, cwd=PROJECT_DIR, env=env)


## 8. Run recommendation


In [ ]:
USER_ID = "U8125"
run([PYTHON_BIN, "-m", "src.run_recommend_cli", "--user", USER_ID, "--topk", "10", "--json"], cwd=PROJECT_DIR)


## 9. Copy artifacts back to Google Drive

Nhung file nay khong nen commit vao Git. Luu trong Drive de tai ve dung local/demo.


In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/news-rec-artifacts")
ARTIFACTS = [
    "models/ctr_model.pt",
    "models/training_report.json",
    "data/precompute/news_embeddings.npy",
    "data/precompute/news_metadata.csv",
    "data/precompute/user_history.json",
    "data/precompute/ctr_dataset.csv",
    "data/precompute/manifest.json",
]

for rel_path in ARTIFACTS:
    source = PROJECT_DIR / rel_path
    target = OUTPUT_DIR / rel_path
    if source.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        print("Copied", source, "->", target)
    else:
        print("Missing artifact, skipped:", source)
